In [4]:
from src.runtime_constants import rr
from src.utils.gembase_utils import GembaseUtils
import json
from src.db import DbConnection

conn = DbConnection()
#conn = DbConnection(rr.ENV_PROD, True)

page = 0
limit = 100

while True:

    print(f"Page: {page}")

    rows_apps = conn.select_all(f"""
    SELECT a.app_id_int, asd.data, a.store
      FROM scraped_data.apps a,
           scraped_data.apps_source_data asd
     WHERE asd.app_id_int = a.app_id_int
       AND (asd.data_type = 'store_page' and a.store = 2
        OR asd.data_type = 'scrap_data' and a.store = 1)
     ORDER BY a.app_id_int
     LIMIT {page * limit}, {limit}
    """)

    if len(rows_apps) == 0:
        break

    page += 1

    bulk_data_del = []
    bulk_data_ins = []

    for row in rows_apps:

        data = GembaseUtils.decompress(row["data"])

        images = []

        if row["store"] == 1:
            data_parsed = json.loads(data)
            if "screenshots" in data_parsed:
                for i in range(len(data_parsed["screenshots"])):
                    images.append(data_parsed["screenshots"][i])
        elif row["store"] == 2:
            ix1 = data.find('id="highlight_strip_scroll"')
            ix2 = data.find('class="slider_ctn"')
            if ix1 == -1 or ix2 == -1:
                continue
            while True:
                img_ix1 = data.find('<img src="', ix1)
                if img_ix1 == -1 or img_ix1 > ix2:
                    break
                img_ix1 += len('<img src="')
                img_ix2 = data.find('">', img_ix1)
                img_url = data[img_ix1:img_ix2]
                img_ix3 = img_url.find("?")
                if img_ix3 != -1:
                    img_url = img_url[:img_ix3]
                img_ix3 = img_url.rfind(".")
                if img_ix3 != -1:
                    img_ix4 = img_url.rfind(".", 0, img_ix3 - 1)
                    if img_ix4 != -1:
                        img_url = img_url[:img_ix4] + img_url[img_ix3:]
                #print(img_url)
                images.append(img_url)
                ix1 = img_ix2

        bulk_data_del.append((row["app_id_int"],))

        for i in range(len(images)):
            bulk_data_ins.append((row["app_id_int"], images[i], i + 1))

    conn.bulk("""
    DELETE FROM scraped_data.apps_gallery ag
    WHERE ag.app_id_int = %s
    """, bulk_data_del)

    conn.bulk("""
    INSERT INTO scraped_data.apps_gallery
    (app_id_int, img_url, img_order)
    VALUES (%s, %s, %s)
    """, bulk_data_ins)

    conn.commit()
conn.close()

Page: 0
Page: 1


In [ ]:
conn.close()